# Warehouse Explorer

Quick exploration of the DuckDB analytics layer.

**Prerequisites:** Run the pipeline first:
```bash
make ingest TICKER=PANW
make warehouse TICKER=PANW
```

The warehouse exposes these views:
| View | Description |
|---|---|
| `v_canonical_facts` | Best filing per (line_item, period, frame) |
| `v_income_statement_quarterly` | IS pivoted wide with provenance triples |
| `v_balance_sheet_quarterly` | BS pivoted wide with provenance triples |
| `v_cash_flow_quarterly` | CF pivoted wide with provenance triples |
| `v_key_metrics` | Margins, YoY/QoQ growth (quarterly) |
| `v_restatement_details` | Rows where a /A amendment materially restates |
| `v_missing_coverage` | (line_item, quarter) pairs with no data |
| `v_data_quality` | One-row summary: has_physical_inventory, has_restatement |

In [ ]:
import sys
from pathlib import Path

import duckdb

# Add repo root to path so src imports work
repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

TICKER = "PANW"  # change to CRWD or SNOW as needed
DB_PATH = repo_root / "data" / "processed" / f"{TICKER}.duckdb"

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"{DB_PATH} not found.\n"
        f"Run:  make ingest TICKER={TICKER} && make warehouse TICKER={TICKER}"
    )

con = duckdb.connect(str(DB_PATH), read_only=True)
print(f"Connected to {DB_PATH}")

In [ ]:
# ── Data quality summary ───────────────────────────────────────────────────
dq = con.execute("SELECT * FROM v_data_quality").fetchdf()
print("v_data_quality:")
display(dq.T.rename(columns={0: "value"}))

In [ ]:
# ── Income statement: quarterly Revenue and margins ────────────────────────
is_q = con.execute("""
    SELECT fiscal_year, fiscal_period, period_end,
           Revenue / 1e9 AS Revenue_B,
           OperatingIncome / 1e9 AS OperatingIncome_B,
           NetIncome / 1e9 AS NetIncome_B,
           revenue_accession
    FROM v_income_statement_quarterly
    WHERE period_type = 'Q'
    ORDER BY fiscal_year, fiscal_period
""").fetchdf()

print(f"Quarterly IS rows: {len(is_q)}")
display(is_q.tail(8))

In [ ]:
# ── Key metrics: margins and growth ───────────────────────────────────────
metrics = con.execute("""
    SELECT fiscal_year, fiscal_period,
           ROUND(gross_margin_pct * 100, 1)     AS gross_margin_pct,
           ROUND(operating_margin_pct * 100, 1) AS operating_margin_pct,
           ROUND(revenue_yoy_growth * 100, 1)   AS revenue_yoy_pct
    FROM v_key_metrics
    ORDER BY fiscal_year, fiscal_period
""").fetchdf()

display(metrics.tail(8))

In [ ]:
# ── Revenue trend chart ────────────────────────────────────────────────────
try:
    import matplotlib.pyplot as plt

    rev = con.execute("""
        SELECT period_end::DATE AS period_end, Revenue / 1e9 AS Revenue_B
        FROM v_income_statement_quarterly
        WHERE period_type = 'Q' AND Revenue IS NOT NULL
        ORDER BY period_end
    """).fetchdf()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(rev["period_end"].astype(str), rev["Revenue_B"], color="steelblue", width=60)
    ax.set_title(f"{TICKER} — Quarterly Revenue ($B)")
    ax.set_xlabel("Quarter end")
    ax.set_ylabel("Revenue ($B)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — skipping chart.")

In [ ]:
# ── Provenance: sample source citations ───────────────────────────────────
# Every Revenue figure can be traced to a specific SEC filing.
prov = con.execute("""
    SELECT period_end, fiscal_year, fiscal_period,
           ROUND(Revenue / 1e9, 3) AS Revenue_B,
           revenue_accession,
           revenue_filing_url
    FROM v_income_statement_quarterly
    WHERE period_type = 'Q' AND Revenue IS NOT NULL
    ORDER BY fiscal_year DESC, fiscal_period DESC
    LIMIT 5
""").fetchdf()

print("Revenue provenance (most recent 5 quarters):")
display(prov)

In [ ]:
# ── Missing coverage ───────────────────────────────────────────────────────
missing = con.execute("""
    SELECT line_item, COUNT(*) AS missing_quarters
    FROM v_missing_coverage
    GROUP BY line_item
    ORDER BY missing_quarters DESC
""").fetchdf()

if missing.empty:
    print("No missing coverage detected.")
else:
    print("Line items with missing quarters:")
    display(missing)

con.close()
print("\nDone.")